# PottsMPNN energy prediction

`run_energy_prediction` runs PottsMPNN's energy model over a structure and returns an
`EnergyPredictionResult`. With no mutant list it performs a **deep mutational scan** (every
position × every non-wild-type amino acid in each chain), which is exactly the data behind a
per-position mutation-energy heatmap.

The result object holds the per-mutation scores as a DataFrame (`.scores`) and can draw the
heatmap on demand with `.plot_heatmap()`.

> PottsMPNN is bundled as the `external/PottsMPNN` submodule; no path configuration is needed.

In [ ]:
from pathlib import Path

# import torch


In [ ]:
import snekwrap.config as config
from snekwrap.wrappers.pottsmpnn import run_energy_prediction

In [ ]:


# Example PDBs ship with the PottsMPNN submodule.
example_pdbs = Path(config.POTTSMPNN_REPO) / "inputs" / "example_pdbs"
pdb = example_pdbs / "3dkm.pdb"  # small single-chain example, fast on CPU

# dev = "cuda" if torch.cuda.is_available() else "cpu"
dev = "cpu"
print(f"running on: {dev}")

## Deep mutational scan

One call: pass the PDB (and any config as keyword arguments). The default model is
`vanilla_model_weights/pottsmpnn_msa_20.pt`; `ddG=True` returns energies relative to the
wild type (wild type centered at 0).

In [ ]:
result = run_energy_prediction(pdb, dev=dev)

print("pdb:", result.pdb_name)
print("chains:", result.chain_order)
print("scored mutants:", len(result.scores))
result.scores.head()

## Heatmap

`plot_heatmap()` draws the [20 amino acids × positions] heatmap and **returns the matplotlib
`Figure`**, so you can customize it (titles, annotations, size) and re-save. Returning it as the
last line of a cell displays it; assign it (`fig = result.plot_heatmap()`) to modify it first.
Pass `save_path` to also write a PNG.

In [ ]:
result.plot_heatmap()

In [ ]:
result.plot_heatmap(save_path="3dkm_ddg_heatmap.png")

In [ ]:
# plot_heatmap returns the Figure, so you can tweak it however you like
fig = result.plot_heatmap()
ax = fig.axes[0]  # the heatmap axes (fig.axes[1] is the colorbar)
ax.set_title("3dkm deep mutational scan", fontsize=14)
fig.set_size_inches(16, 4)
fig  # redisplay the modified figure

## Multi-chain complexes

For a complex you can restrict the scan to particular chains with `exclude_chains`, and zoom
the heatmap to a position range per chain with `chain_ranges` (inclusive, 1-indexed).

`6w25` is a peptide (chain A) bound to a protein (chain B); here we scan only chain B.

In [ ]:
complex_result = run_energy_prediction(
    example_pdbs / "6w25.pdb",
    exclude_chains=["A"],  # scan chain B only
    dev=dev,
)

print("chains:", complex_result.chain_order, "| scored mutants:", len(complex_result.scores))

# zoom the heatmap to chain B positions 50-100
complex_result.plot_heatmap(chain_ranges={"B": [50, 100]})

## Binding energy

To predict how mutations affect **binding** instead of stability, pass `binding_energy_json` —
a mapping from the PDB name to the chain partitions of the complex (its binding partners). The
predicted value then becomes the change in binding energy, E(bound) − Σ E(unbound partitions),
so positive values indicate mutations that weaken binding.

Here we scan the `6w25` peptide (chain A) and split the complex into the peptide `[A]` and its
receptor `[B]`. Since only chain A is scanned, we pass `only_mutated_positions=True` to
`plot_heatmap` so the heatmap shows just the scanned positions instead of every position in the
(much larger) receptor chain.

In [ ]:
binding_result = run_energy_prediction(
    example_pdbs / "6w25.pdb",
    exclude_chains=["B"],                          # scan the peptide (chain A)
    binding_energy_json={"6w25": [["A"], ["B"]]},  # complex partitions: peptide [A] vs receptor [B]
    dev=dev,
)
print("binding ΔΔG predictions:", len(binding_result.scores))

# Only the peptide (chain A) was scanned, so use only_mutated_positions=True to drop the
# hundreds of un-scanned chain-B columns (otherwise the heatmap is mostly empty).
binding_result.plot_heatmap(
    only_mutated_positions=True,
    title="PottsMPNN binding prediction",
    clabel=r"Predicted binding $\Delta\Delta$G (a.u.)",
)